# SARIMA (Seasonal AutoRegressive Integrated Moving Average)

While standard **ARIMA** can handle overall upward or downward trends, it has a fundamental blind spot: **it cannot model repeating cyclical patterns (seasonality)**.

Real-world time series almost always repeat patterns at fixed intervals:
* **Retail & E-commerce:** Massive sales surges every December (holiday shopping).
* **Energy & Electricity Demand:** Peaks every summer (air conditioning) and during evening hours.
* **Travel & Tourism:** Passenger volume surges every July/August and drops in February.

**SARIMA** extends ARIMA by adding dedicated seasonal components to model repeating cyclical waves while simultaneously handling non-seasonal trends and noise.

## The SARIMA Notation: $(p, d, q)(P, D, Q)_s$

A SARIMA model is characterized by **7 distinct parameters**, grouped into non-seasonal and seasonal components:

$$\text{SARIMA}\underbrace{(p, d, q)}_{\text{Non-Seasonal Orders}} \times \underbrace{(P, D, Q)_s}_{\text{Seasonal Orders}}$$

### 1. Non-Seasonal Parameters $(p, d, q)$
These model short-term dependencies between **consecutive time steps** (e.g., today vs. yesterday):
* **$p$ (AutoRegressive):** Number of recent lag observations used to predict the current value.
* **$d$ (Differencing):** Number of times consecutive values are subtracted to remove the general upward or downward trend ($\Delta Y_t = Y_t - Y_{t-1}$).
* **$q$ (Moving Average):** Number of recent forecast error terms (shocks) used to adjust the forecast.

### 2. Seasonal Parameters $(P, D, Q)_s$
These model dependencies between **the same period across repeating cycles** (e.g., this July vs. last July):
* **$s$ (Seasonal Period Length):** Number of time steps per complete seasonal cycle:
  * $s = 12$ for monthly data with annual cycles (e.g., seasonal airline traffic).
  * $s = 4$ for quarterly business data (e.g., Q4 holiday revenue).
  * $s = 7$ for daily data with weekly cycles (e.g., weekend retail spikes).
  * $s = 24$ for hourly data with daily cycles (e.g., electricity load curves).
* **$P$ (Seasonal AR):** Number of past seasonal periods used (e.g., using sales from 12 or 24 months ago).
* **$D$ (Seasonal Differencing):** Subtracting the value from the exact same season in the previous cycle ($\Delta_s Y_t = Y_t - Y_{t-s}$).
* **$Q$ (Seasonal MA):** Number of seasonal forecast shocks/errors incorporated from past cycles.

## How Seasonal Differencing ($D$) Works

Just as non-seasonal differencing ($d = 1$) removes general secular trends, **seasonal differencing ($D = 1$)** eliminates repeating cyclical wave shapes.

Instead of subtracting yesterday from today, it subtracts the observation from the exact same season in the preceding cycle:

$$\Delta_s Y_t = Y_t - Y_{t-s}$$

### Monthly Example ($s = 12$)
$$\Delta_{12} Y_{\text{Dec 2024}} = Y_{\text{Dec 2024}} - Y_{\text{Dec 2023}}$$

* If retail sales predictably surge by 40% every December, subtracting last December's sales strips away that predictable holiday bump.
* In practice, **$D = 1$** is almost always sufficient to remove seasonal cycles; $D > 1$ is virtually never required.

## An Intuitive Example: Ice Cream Sales Across Years

Consider an ice cream brand tracking monthly sales over multiple years:

```text
Sales
  ▲            /\                     /\
  │           /  \                   /  \
  │          /    \                 /    \
  │    /\   /      \          /\   /      \
  │   /  \_/        \        /  \_/        \
  └───┴──────┴───────┴───────┴──────┴───────┴──► Time
     Jan    Jul     Dec     Jan    Jul     Dec
           Year 1                 Year 2
```

To model this pattern accurately, SARIMA orchestrates four distinct mechanisms:

1. **Overall Long-Term Trend (Handled by $d$):**
   Because the business is growing year-over-year, the Year 2 peaks are higher than Year 1. First-order differencing ($d = 1$) removes this upward drift.

2. **Repeating Seasonal Wave (Handled by $D = 1$ and $s = 12$):**
   Sales consistently peak in July (summer) and hit troughs in January (winter). Seasonal differencing ($D = 1$) removes this annual sinusoidal wave.

3. **Short-Term Month-to-Month Dynamics (Handled by $p, q$):**
   If June was unusually warm, July sales get an immediate short-term momentum boost ($p = 1$).

4. **Year-Over-Year Seasonal Shocks (Handled by $P, Q$):**
   If an extreme heatwave last summer caused an unexpected demand surge, the seasonal moving average term ($Q = 1$) carries forward that seasonal surprise to adjust this summer's expectations.

## Mathematical Representation of SARIMA & Formula Explainability

Using the compact **backshift / lag operator** $B$ (where $B^k Y_t = Y_{t-k}$), the full unified SARIMA equation is written as:

$$\underbrace{\Phi_P(B^s)}_{\text{Seasonal AR}} \underbrace{\phi_p(B)}_{\text{Non-Seasonal AR}} \underbrace{(1 - B^s)^D}_{\text{Seasonal Differencing}} \underbrace{(1 - B)^d}_{\text{Non-Seasonal Differencing}} Y_t = \underbrace{c}_{\text{Baseline (Constant)}} + \underbrace{\Theta_Q(B^s)}_{\text{Seasonal MA}} \underbrace{\theta_q(B)}_{\text{Non-Seasonal MA}} \epsilon_t$$

### Detailed Breakdown of Every Component

#### 1. Baseline (Constant / Drift Term, $c$)
* **Where it sits:** On the right-hand side as an additive constant $c$.
* **Role:** It provides the deterministic baseline or drift rate of the process. In differenced time series ($d=1$), $c$ represents the expected permanent growth or decline per time step.

#### 2. Non-Seasonal Autoregressive (AR) Component: $\phi_p(B)$
* **Symbol:** Lowercase Greek letter phi: $\phi_p(B)$
* **Expanded form:** $(1 - \phi_1 B - \phi_2 B^2 - \dots - \phi_p B^p)$
* **Role:** Captures the direct relationship between today's value and **immediate preceding days** ($Y_{t-1}, Y_{t-2}, \dots$). It models short-term momentum and daily behavioral habit.

#### 3. Non-Seasonal Moving Average (MA) Component: $\theta_q(B)$
* **Symbol:** Lowercase Greek letter theta: $\theta_q(B)$
* **Expanded form:** $(1 + \theta_1 B + \theta_2 B^2 + \dots + \theta_q B^q)$
* **Role:** Adjusts today's forecast using recent **consecutive prediction shocks/errors** ($\epsilon_{t-1}, \epsilon_{t-2}, \dots$). It acts as a short-term shock absorber for random unexpected events.

#### 4. Seasonal Autoregressive (SAR) Component: $\Phi_P(B^s)$
* **Symbol:** Uppercase Greek letter Phi: $\Phi_P(B^s)$
* **Expanded form:** $(1 - \Phi_1 B^s - \Phi_2 B^{2s} - \dots - \Phi_P B^{Ps})$
* **Role:** Uses past values from the **exact same season in previous cycles** (e.g., comparing this July to last July, $Y_{t-s}$, or two years ago, $Y_{t-2s}$). It models multi-cycle recurring memory.

#### 5. Seasonal Moving Average (SMA) Component: $\Theta_Q(B^s)$
* **Symbol:** Uppercase Greek letter Theta: $\Theta_Q(B^s)$
* **Expanded form:** $(1 + \Theta_1 B^s + \dots + \Theta_Q B^{Qs})$
* **Role:** Incorporates unexpected **forecast shocks that occurred in the same season in prior cycles** (e.g., an abnormal spike during last Christmas, $\epsilon_{t-s}$, influencing adjustments for this Christmas).

#### 6. Non-Seasonal Differencing: $(1 - B)^d$
* **Expanded form for $d=1$:** $(1 - B)Y_t = Y_t - Y_{t-1}$
* **Role:** Subtracts consecutive periods to eliminate linear or secular trends, stabilizing the series' mean over time.

#### 7. Seasonal Differencing: $(1 - B^s)^D$
* **Expanded form for $D=1$:** $(1 - B^s)Y_t = Y_t - Y_{t-s}$
* **Role:** Subtracts values separated by an entire seasonal period $s$ (e.g., $Y_{\text{Dec 2024}} - Y_{\text{Dec 2023}}$) to strip away predictable cyclical waves and seasonal baselines.

## How to Identify and Select SARIMA Orders

Selecting orders for SARIMA requires checking both non-seasonal lags and seasonal lag multiples:

### Step 1: Determine the Seasonal Period $s$
Identify the frequency of the dataset:
* Monthly data with annual recurrence $\to s = 12$
* Daily data with day-of-week patterns $\to s = 7$
* Quarterly data with annual recurrence $\to s = 4$

### Step 2: Determine Differencing Orders ($d$ and $D$)
1. Apply Augmented Dickey-Fuller (ADF) test on the raw data. If non-stationary, set $d = 1$.
2. If strong spikes persist at seasonal lags ($s, 2s, 3s...$) in the ACF plot after non-seasonal differencing, apply seasonal differencing ($D = 1$).

### Step 3: Determine Non-Seasonal ($p, q$) and Seasonal ($P, Q$) Orders
* **Non-Seasonal Orders (Inspect initial lags 1, 2, 3...):**
  * PACF cutoff at lag $k$ suggests $p = k$.
  * ACF cutoff at lag $k$ suggests $q = k$.
* **Seasonal Orders (Inspect seasonal lag multiples $s, 2s, 3s...$):**
  * PACF cutoff at lag $s$ or $2s$ suggests $P = 1$ or $P = 2$.
  * ACF cutoff at lag $s$ or $2s$ suggests $Q = 1$ or $Q = 2$.

## Differences Between ARIMA and SARIMA: How Good or Bad It Is Compared to ARIMA

### Side-by-Side Feature Comparison

| Dimension | ARIMA$(p, d, q)$ | SARIMA$(p, d, q)(P, D, Q)_s$ |
| :--- | :--- | :--- |
| **Parameters** | 3 parameters: $(p, d, q)$ | 7 parameters: $(p, d, q) \times (P, D, Q)_s$ |
| **Timescale Perspective** | Single timescale (consecutive time steps) | Dual timescale (step-to-step AND season-to-season) |
| **Handles Secular Trends?** | **Yes** (via non-seasonal differencing $d$) | **Yes** (via non-seasonal differencing $d$) |
| **Handles Repeating Cycles?** | **No** (treats seasonal spikes as unmodeled error) | **Yes** (models recurring cycles via $P, D, Q, s$) |
| **Total Differencing** | $\Delta Y_t = Y_t - Y_{t-1}$ | $\Delta \Delta_s Y_t = (Y_t - Y_{t-1}) - (Y_{t-s} - Y_{t-s-1})$ |
| **Data Volume Needed** | Moderate (e.g., 30–50 observations) | High (requires at least 3–5 full seasonal cycles) |
| **Tuning Complexity** | Low ($O(p \times d \times q)$) | High ($O(p \cdot d \cdot q \cdot P \cdot D \cdot Q)$) |

### How Good SARIMA Is Compared to ARIMA (When SARIMA Wins)

* **Preserves Cyclical Shape and Amplitude:**
  When seasonal patterns exist (e.g., electricity peaks in July or retail peaks in December), ARIMA produces flatline forecasts or continues a straight drift, missing every single peak and trough. SARIMA reproduces the exact periodic oscillations.

* **Drastic Error Reduction (30% to 70% Lower RMSE/MAE):**
  On seasonal data, ARIMA leaves strong cyclical patterns trapped in the residuals, causing massive forecast errors. SARIMA extracts this seasonal structure, drastically reducing Root Mean Squared Error (RMSE) and Mean Absolute Error (MAE).

* **Statistically Sound Residuals (White Noise):**
  Because ARIMA cannot capture seasonal lags, its residuals exhibit strong periodic autocorrelation and fail diagnostic tests (like the Ljung-Box test). SARIMA residuals clean out seasonal autocorrelation, producing true white noise.

### How Bad SARIMA Is Compared to ARIMA (When ARIMA Wins & SARIMA's Downsides)

* **High Risk of Overfitting on Non-Seasonal Data:**
  If the underlying data does not possess genuine seasonality (e.g., stock price fluctuations or random walks), forcing seasonal terms $(P, D, Q)$ into the model fits noise rather than signal, hurting out-of-sample accuracy.

* **Exponentially Larger Hyperparameter Search Space:**
  Tuning ARIMA requires searching only 3 parameters $(p, d, q)$. Tuning SARIMA requires searching 7 parameters. An automated grid search (`auto_arima`) that takes 2 seconds for ARIMA can take minutes or hours for SARIMA.

* **Strict Data Length Requirements:**
  To accurately estimate seasonal parameters $(P, D, Q)$, you need at least 3 to 5 full cycles of data. For monthly data with annual seasonality ($s = 12$), that means a minimum of 36 to 60 consecutive months. ARIMA can be fit on much shorter sequences.

* **Danger of Over-Differencing ($d + D > 2$):**
  Applying both non-seasonal differencing ($d = 1$) and seasonal differencing ($D = 1$) doubles the differencing operations. If not strictly needed, this injects artificial negative autocorrelation and inflates model variance.

## Advantages and Disadvantages of SARIMA (Pros & Cons)

### Advantages (Pros) of SARIMA

* **Comprehensive Dual-Level Forecasting:**
  Simultaneously captures immediate day-to-day momentum, long-term secular growth trends, and annual or weekly recurring cycles in a single unified statistical model.

* **Theoretical Rigor & Interpretability:**
  Rooted in the disciplined Box-Jenkins framework. Every parameter has clear mathematical meaning, and confidence intervals are derived from formal probability theory.

* **Dynamic Multi-Cycle Shock Absorption:**
  The seasonal moving average ($Q$) accommodates shocks that happen at specific times of the year (e.g., an unusually severe winter storm last January), allowing the model to adapt without distorting normal monthly behavior.

* **No Need for Dummy Seasonality Columns:**
  Unlike standard linear regression models that require adding 11 one-hot encoded dummy variables for monthly seasonality, SARIMA handles cyclic patterns natively through difference and lag operators.

### Disadvantages (Cons) & Limitations of SARIMA

* **Single Seasonal Period Limitation:**
  SARIMA can only handle **one seasonal period $s$** at a time. It cannot natively model datasets with complex multiple seasonalities (e.g., hourly electricity data that has a 24-hour daily cycle, a 7-day weekly cycle, and a 365-day annual cycle). Modern models like **Prophet** or **TBATS** are required for multiple seasonalities.

* **Heavy Computational Overhead:**
  Evaluating likelihood functions and inverting matrices across high seasonal lags (e.g., lag 12 or 24) makes training and grid-searching noticeably slower.

* **Extreme Sensitivity to Missing Values:**
  Because seasonal differencing calculates $Y_t - Y_{t-s}$, a missing observation at timestamp $t-s$ ruins the differencing calculation for timestamp $t$, requiring imputation before model fitting.

* **Linearity Restriction:**
  Like ARIMA, SARIMA assumes linear relationships and cannot model non-linear seasonal interactions or sudden structural regime shifts.

## How to Implement SARIMA in Python

In Python, SARIMA models are implemented using `statsmodels.tsa.statespace.sarimax.SARIMAX`:

```python
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX
import pmdarima as pm

%matplotlib inline

# 1. Load classic AirPassengers dataset
url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv'
df = pd.read_csv(url, index_col='Month', parse_dates=True)
df.index.freq = 'MS'

# 2. Train/Test split
train = df.iloc[:-12]
test = df.iloc[-12:]

# 3. Automated Order Selection via Auto-ARIMA with Seasonality (m=12)
auto_sarima = pm.auto_arima(
    train,
    seasonal=True,
    m=12,
    start_p=0, max_p=2,
    start_q=0, max_q=2,
    d=1, D=1,
    trace=True,
    error_action='ignore',
    suppress_warnings=True
)
print('Optimal SARIMA Order:', auto_sarima.order, auto_sarima.seasonal_order)

# 4. Fit SARIMAX model
model = SARIMAX(
    train,
    order=auto_sarima.order,
    seasonal_order=auto_sarima.seasonal_order,
    enforce_stationarity=False,
    enforce_invertibility=False
)
fitted_model = model.fit()

# 5. Generate Forecast with 95% Confidence Intervals
forecast_res = fitted_model.get_forecast(steps=len(test))
pred_mean = forecast_res.predicted_mean
conf_int = forecast_res.conf_int()

# 6. Plot Results
plt.figure(figsize=(12, 5))
plt.plot(train.index, train, label='Training Data')
plt.plot(test.index, test, label='Actual Test Data', color='green')
plt.plot(test.index, pred_mean, label='SARIMA Forecast', color='red', linestyle='--')
plt.fill_between(test.index, conf_int.iloc[:, 0], conf_int.iloc[:, 1], color='pink', alpha=0.3, label='95% CI')
plt.title('SARIMA Forecast vs. Actuals')
plt.xlabel('Date')
plt.ylabel('Passengers')
plt.legend()
plt.show()
```